#### Weather Data Ingestion — Open-Meteo Archive API

The weather layer is fetched from the **Open-Meteo Archive API** — a free, no-authentication historical weather service providing hourly granularity for any coordinate. Berlin's central coordinates (52.52°N, 13.41°E) are used as a single city-wide reference point, as district-level weather variation across Mitte, Prenzlauer Berg, and Friedrichshain-Kreuzberg is negligible for operational leakage analysis.

**Time Window Alignment:**
The fetch window mirrors the 3-week trip log timeline. The end date is offset by 5 days to account for the archive API's inherent data lag, ensuring no incomplete or empty hourly rows are returned at the tail of the dataset.

**Metrics:**
Only `temperature_celsius` and `rain_mm` are retained as analytically meaningful leakage signals — cold temperatures and precipitation are the primary environmental drivers of demand surges and idle clustering in urban ride-sharing contexts.

**Seasonal Scoping:**
`snowfall_cm` is explicitly excluded as this pipeline is scoped to spring/summer operations (Q2/Q3), during which snowfall returns zero values and adds no signal. The parameter can be re-enabled for Q4 seasonal analysis without structural changes to the pipeline.

**Join Strategy:**
The weather DataFrame uses `timestamp_hour` rather than `timestamp` to avoid column collision during the merge with trip logs. Trip timestamps are floored to the nearest hour (`dt.floor("H")`) to produce a clean join key, preserving the original minute-level granularity of the trip data.

In [3]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# 1. Define the parameters for central Berlin coordinates
LATITUDE = 52.5200
LONGITUDE = 13.4050

# 2. Calculate the exact 3-week time window matching your trips
# End date is offset by 5 days to account for the archive API's data lag
end_date = datetime.now() - timedelta(days=5)
start_date = end_date - timedelta(days=21)
start_str = start_date.strftime('%Y-%m-%d')
end_str = end_date.strftime('%Y-%m-%d')

# 3. Construct the Open-Meteo Archive API URL
url = (
    f"https://archive-api.open-meteo.com/v1/archive?"
    f"latitude={LATITUDE}&longitude={LONGITUDE}&"
    f"start_date={start_str}&end_date={end_str}&"
    f"hourly=temperature_2m,rain,snowfall&timezone=Europe%2FBerlin"
)

print(f"Fetching real historical weather from {start_str} to {end_str}:")

# 4. Make the live API request
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    hourly_data = data['hourly']

    # 5. Convert JSON response into a clean Pandas DataFrame
    # timestamp_hour is named explicitly to avoid column collision on merge with trip data
    df_weather = pd.DataFrame({
        'timestamp_hour': hourly_data['time'],
        'temperature_celsius': hourly_data['temperature_2m'],
        'rain_mm': hourly_data['rain']
        # snowfall_cm excluded — pipeline scoped to spring/summer operations.
        # Re-enable for Q4 seasonal analysis.    
    })

    # Standardize timestamp format to match the DF: berlin_taxi_trips_raw
    df_weather['timestamp_hour'] = pd.to_datetime(df_weather['timestamp_hour'])

    # 6. Save to CSV
    api_csv_weather = 'berlin_weather_raw.csv'
    df_weather.to_csv(api_csv_weather, index=False)

    #print(f"Success! Saved {len(df_weather)} hourly weather records to '{api_csv_weather}'")
    print("\nFirst 5 rows preview:")
    print(df_weather.head())

else:
    print(f"Failed to fetch data. API Error Code: {response.status_code}")
    print(f"Response: {response.text}")

df_weather

Fetching real historical weather from 2026-05-14 to 2026-06-04:

First 5 rows preview:
       timestamp_hour  temperature_celsius  rain_mm
0 2026-05-14 00:00:00                  7.8      0.1
1 2026-05-14 01:00:00                  7.1      0.9
2 2026-05-14 02:00:00                  6.8      0.0
3 2026-05-14 03:00:00                  6.7      1.2
4 2026-05-14 04:00:00                  6.6      0.2


,timestamp_hour,temperature_celsius,rain_mm
0,2026-05-14 00:00:00,7.8,0.1
1,2026-05-14 01:00:00,7.1,0.9
2,2026-05-14 02:00:00,6.8,0.0
3,2026-05-14 03:00:00,6.7,1.2
4,2026-05-14 04:00:00,6.6,0.2
...,...,...,...
523,2026-06-04 19:00:00,20.8,0.0
524,2026-06-04 20:00:00,21.2,0.0
525,2026-06-04 21:00:00,20.2,0.0
526,2026-06-04 22:00:00,19.0,0.0
